# 07 — Generazione di Recensioni Sintetiche con T5

Questo notebook implementa il fine-tuning di **T5-small** (`t5-small`) per la generazione condizionata di recensioni.
L'obiettivo è generare testo sintetico (Titolo e Recensione) a partire da due condizionamenti:
1. Il nome del prodotto
2. Il rating desiderato (da 1 a 5 stelle)

**Perché T5?**
- È un modello Text-to-Text Transfer Transformer: tratta ogni problema di NLP come una conversione da testo a testo.
- Ideale per la generazione di testo controllata.
- Pre-trained su grandi corpora in lingua inglese.


In [9]:
import sys
sys.path.insert(0, '..')

import os
import re
import json
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from tqdm import tqdm
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer,
    EarlyStoppingCallback
)
import evaluate

from src.data_loader import load_amazon_beauty_reviews, load_amazon_beauty_metadata

## 1. Caricamento del dataset e dei metadati

In [10]:
# Carica le recensioni e i metadati utilizzando le funzioni in src.data_loader
df_reviews = load_amazon_beauty_reviews(cache_dir="../data")
df_metadati = load_amazon_beauty_metadata(cache_dir="../data")

📂 Caricamento dataset da cache locale: ../data\amazon_beauty_reviews.csv
✅ Dataset caricato: 701528 righe, 10 colonne
📂 Caricamento metadati da cache locale: ../data\amazon_beauty_metadati.csv
✅ Metadati caricati: 112590 righe, 16 colonne


## 2. Pulizia e Unione del Dataset

In [11]:
# 1. Rimpiazza tutte le forme testuali di null con np.nan
for col in ["text", "title"]:
    df_reviews[col] = df_reviews[col].replace(to_replace=["nan", "NaN", "None", "null", "NULL", ""], value=np.nan)

# 2. Rimuovi righe con valori nulli reali
df_reviews = df_reviews.dropna(subset=["text", "title", "rating"])

# 3. Convertili in stringa
df_reviews["text"] = df_reviews["text"].astype(str)
df_reviews["title"] = df_reviews["title"].astype(str)

# 4. Elimina righe con solo spazi (o vuote anche dopo strip)
df_reviews = df_reviews[(df_reviews["text"].str.strip() != "") & (df_reviews["title"].str.strip() != "")]

# 5. Rimuovi righe con lunghezza < 5 caratteri per il testo e < 3 per il titolo
df_reviews = df_reviews[(df_reviews["text"].str.len() > 5) & (df_reviews["title"].str.len() > 3)]

# 6. Rimozione duplicati sul campo 'text' (case insensitive)
df_reviews["text_clean"] = df_reviews["text"].str.strip().str.lower()
df_reviews = df_reviews.drop_duplicates(subset=["text_clean"])
df_reviews = df_reviews.drop(columns=["text_clean"])

# 7. Reset dell'indice
df_reviews.reset_index(drop=True, inplace=True)

print(f"Righe rimanenti dopo pulizia recensioni: {len(df_reviews)}")

Righe rimanenti dopo pulizia recensioni: 633861


In [12]:
# Unisce le recensioni con i metadati sul campo parent_asin
df_merged = df_reviews.merge(df_metadati, on="parent_asin", how="left")

# Rinomina le colonne per chiarezza
df_merged = df_merged.rename(columns={"title_y": "product_name", "title_x": "title"})

# Seleziona solo le colonne di interesse
df_merged = df_merged[["rating", "product_name", "title", "text"]]

# Rimuovi righe con product_name nullo
df_merged = df_merged.dropna(subset=["product_name"]).reset_index(drop=True)

print(f"Righe dopo unione metadati: {len(df_merged)}")

Righe dopo unione metadati: 633785


In [13]:
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)  # Rimuove tag HTML
    text = re.sub(r'[^A-Za-z0-9\s.,!?\'"]+', '', text)  # Rimuove caratteri speciali
    return text.strip()

# Applica pulizia e converte in minuscolo
df_merged['title'] = df_merged['title'].astype(str).apply(clean_text).str.lower()
df_merged['text'] = df_merged['text'].astype(str).apply(clean_text).str.lower()
df_merged['product_name'] = df_merged['product_name'].astype(str).str.lower()

In [14]:
def stratified_undersample(df, group_col, text_col, target_count):
    result = []

    for label, group in df.groupby(group_col):
        group = group.copy()
        group["length"] = group[text_col].apply(len)
        group_sorted = group.sort_values("length")

        # Definisci indici per dividere in terzi (corte, medie, lunghe)
        short_range = group_sorted.iloc[:len(group_sorted)//3]
        medium_range = group_sorted.iloc[len(group_sorted)//3:2*len(group_sorted)//3]
        long_range = group_sorted.iloc[2*len(group_sorted)//3:]

        # Calcola le dimensioni con fallback al min(len, target)
        n_short = min(int(target_count * 0.3), len(short_range))
        n_medium = min(int(target_count * 0.4), len(medium_range))
        n_long = min(target_count - n_short - n_medium, len(long_range))

        short = short_range.sample(n=n_short, random_state=42)
        medium = medium_range.sample(n=n_medium, random_state=42)
        long = long_range.sample(n=n_long, random_state=42)

        balanced = pd.concat([short, medium, long])
        result.append(balanced)

    return pd.concat(result).drop(columns=["length"]).reset_index(drop=True)

# Bilancia le classi basandosi sulla classe minoritaria
min_count = df_merged["rating"].value_counts().min()
df_balanced = stratified_undersample(df_merged, group_col="rating", text_col="text", target_count=min_count)

print(f"Dataset bilanciato: {len(df_balanced)} righe totali ({min_count} per ciascuno dei 5 rating)")

Dataset bilanciato: 201786 righe totali (40628 per ciascuno dei 5 rating)


In [15]:
# Prepariamo gli input e target nel formato atteso da T5
# input_text:  Product: <nome>. Rating: <n> stars.
# target_text: Title: <titolo>\nReview: <testo>

df_balanced["input_text"] = df_balanced.apply(
    lambda row: f"Product: {row['product_name']}. Rating: {int(row['rating'])} stars.", axis=1)

df_balanced["target_text"] = df_balanced.apply(
    lambda row: f"Title: {row['title']}\nReview: {row['text']}", axis=1)

dataset_df = df_balanced[["input_text", "target_text"]]
dataset_df.head()

,input_text,target_text
0,Product: broadway nails press-on manicure desi...,Title: one star\nReview: would hold up better ...
1,Product: you're so fine liquid eyeliner cruelt...,Title: one star\nReview: peels off after an hour.
2,Product: rose quartz roller | pink jade roller...,Title: my angel came broken\nReview: besides t...
3,"Product: hot sale!!1pc sexy lingerie,woaills w...",Title: not sized correctly.\nReview: product i...
4,Product: jordana color envy waterproof liquid ...,Title: i wouldn't waste my money\nReview: it d...


## 3. Fine-tuning di T5-small

In [16]:
# 1. Carica tokenizer e modello T5-small
tokenizer = AutoTokenizer.from_pretrained("t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 2. Crea HuggingFace Dataset
dataset = Dataset.from_pandas(dataset_df)
dataset = dataset.map(lambda x: {
    "input_text": "generate: " + x["input_text"],
    "target_text": x["target_text"]
})

# 3. Split train/test
dataset_split = dataset.train_test_split(test_size=0.3, seed=42)
train_dataset = dataset_split["train"]
test_dataset = dataset_split["test"]

# 4. Tokenizzazione
def tokenize(example):
    model_inputs = tokenizer(
        example["input_text"],
        max_length=256,
        padding="max_length",
        truncation=True
    )
    labels = tokenizer(
        example["target_text"],
        max_length=256,
        padding="max_length",
        truncation=True
    )["input_ids"]
    model_inputs["labels"] = [
        (token if token != tokenizer.pad_token_id else -100)
        for token in labels
    ]
    return model_inputs

tokenized_train = train_dataset.map(tokenize, batched=True)
tokenized_test = test_dataset.map(tokenize, batched=True)
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

# 5. Iperparametri definiti localmente (params.yaml rimosso come richiesto)
t5_params = {
    "batch_size_train": 8,
    "batch_size_eval": 8,
    "epochs": 3,
    "learning_rate": 5e-5,
    "weight_decay": 0.01,
    "warmup_steps": 100,
    "lr_scheduler_type": "linear"
}

# 6. Configurazione TrainingArguments
training_args = Seq2SeqTrainingArguments(
    output_dir="../models/t5-small-finetuned",
    per_device_train_batch_size=t5_params["batch_size_train"],
    per_device_eval_batch_size=t5_params["batch_size_eval"],
    gradient_accumulation_steps=1,
    num_train_epochs=t5_params["epochs"],
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir="../models/t5-small-finetuned/logs",
    logging_steps=20,
    load_best_model_at_end=True,
    report_to="none",
    metric_for_best_model="loss",
    fp16=torch.cuda.is_available(),
    learning_rate=t5_params["learning_rate"],
    weight_decay=t5_params["weight_decay"],
    warmup_steps=t5_params["warmup_steps"],
    lr_scheduler_type=t5_params["lr_scheduler_type"]
)

# 7. Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# Avvia training (tracciamento MLflow rimosso)
trainer.train()

# 8. Salva modello e tokenizer finali
model_save_path = "../models/t5-small-finetuned/final_model"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("[SAVE] Modello e tokenizer salvati correttamente in: " + model_save_path)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Map:   0%|          | 0/201786 [00:00<?, ? examples/s]

Map:   0%|          | 0/141250 [00:00<?, ? examples/s]

Map:   0%|          | 0/60536 [00:00<?, ? examples/s]

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
c:\Users\frape\Desktop\UNISA\NLP\ReactNLP\NLP-REACT\.venv\Lib\site-packages\transformers\data\data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[SAVE] Modello e tokenizer salvati correttamente in: ../models/t5-small-finetuned/final_model


## 4. Test di inferenza

In [17]:
# Carica modello e tokenizer salvati
model_dir = "../models/t5-small-finetuned/final_model"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

# Sposta modello su GPU se disponibile
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Esempio di input nello stesso formato usato in training
product_name = "anti-aging face cream"
rating = 5
input_prompt = f"generate: Product: {product_name}. Rating: {rating} stars."

# Tokenizza e genera
inputs = tokenizer(input_prompt, return_tensors="pt", truncation=True).to(device)
outputs = model.generate(
    **inputs,
    max_length=128,
    do_sample=True,
    top_k=50,
    top_p=0.95,
    temperature=0.7,
    repetition_penalty=2.0
)

# Decodifica
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"[INPUT] Input:\n{input_prompt}\n")
print(f"[OUTPUT] Recensione generata:\n{generated_text}")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

[INPUT] Input:
generate: Product: anti-aging face cream. Rating: 5 stars.

[OUTPUT] Recensione generata:
Title: five stars Review: i love this product!


## 5. Generazione in Batch per la Valutazione

In [23]:
# Prepara gli input e reference per il test set
inputs_test = test_dataset["input_text"]
references_test = test_dataset["target_text"]

batch_size = 64
generated = []

generated_path = "../results/generated.json"
references_path = "../results/references.json"

if not os.path.exists(generated_path):
    print("[GENERATING] Avvio generazione in batch...")
    for i in tqdm(range(0, len(inputs_test), batch_size)):
        batch_inputs = inputs_test[i:i+batch_size]

        batch_tokenized = tokenizer(
            batch_inputs,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids=batch_tokenized["input_ids"],
                attention_mask=batch_tokenized["attention_mask"],
                max_new_tokens=64,
                num_beams=4,
                do_sample=False
            )

         # Decodifica
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        generated.extend(decoded)

    # Salva i testi generati e le reference nella cartella results
    os.makedirs("../results", exist_ok=True)
    with open(generated_path, "w", encoding="utf-8") as f:
        json.dump(generated, f, ensure_ascii=False, indent=2)

    wrapped_references = [[ref] for ref in references_test]
    with open(references_path, "w", encoding="utf-8") as f:
        json.dump(wrapped_references, f, ensure_ascii=False, indent=2)

    print("[SUCCESS] Generazione completata e salvata!")
    with open(generated_path, "r", encoding="utf-8") as f:
        generated = json.load(f)
    with open(references_path, "r", encoding="utf-8") as f:
        references = json.load(f)
else:
    print("[CACHE] File di generazione trovati nella cache locale.")
    with open(generated_path, "r", encoding="utf-8") as f:
        generated = json.load(f)
    with open(references_path, "r", encoding="utf-8") as f:
        references = json.load(f)

[CACHE] File di generazione trovati nella cache locale.


## 6. Valutazione Quantitativa (BERTScore)

In [24]:
# Calcola BERTScore
bertscore = evaluate.load("bertscore")
clean_references = [ref[0] if isinstance(ref, list) else ref for ref in references]
bertscore_result = bertscore.compute(
    predictions=generated,
    references=clean_references,
    lang="en"
)

# Calcola la media dei punteggi F1
avg_bertscore = sum(bertscore_result["f1"]) / len(bertscore_result["f1"])
print(f"BERTScore (F1): {avg_bertscore:.4f}")

# Salva la metrica in results/t5_metrics.json
with open("../results/t5_metrics.json", "w") as f:
    json.dump({"bertscore_f1": avg_bertscore}, f, indent=2)

print("[SAVE] Metriche salvate in: ../results/t5_metrics.json")

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1): 0.8623
[SAVE] Metriche salvate in: ../results/t5_metrics.json
